# minilink in one notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/basics/01_minilink.ipynb)

Write a model, simulate it, connect a controller and animate the loop: the whole basic workflow.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Write your own model

A model is a class with two equations and a dictionary of parameters:

$$\dot x = f(x, u, t), \qquad y = h(x, u, t)$$

Subclass `DynamicSystem`, give the dimensions to the constructor, store the constants in
`params`, and write `f` and `h` as in the textbook. Here is a mass-spring-damper,
$m \ddot q = F - k q - c \dot q$, with the state $x = [q, \dot q]$, the force $F$ as
input and the position $q$ as output.

In [ ]:
import numpy as np

from minilink import DynamicSystem


class MassSpringDamper(DynamicSystem):
    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=1)
        self.params = {"m": 1.0, "k": 4.0, "c": 0.4}
        self.x0 = np.array([1.0, 0.0])

    def f(self, x, u, t=0.0, params=None):
        params = self.params if params is None else params
        m, k, c = params["m"], params["k"], params["c"]
        q, dq = x[0], x[1]
        F = u[0]
        ddq = (F - k * q - c * dq) / m
        return np.array([dq, ddq])

    def h(self, x, u, t=0.0, params=None):
        return x[0:1]

`compute_trajectory` integrates `f` from the initial state `x0` and returns the sampled
`Trajectory`; `plot_trajectory` draws it. With no input connected, `u` stays at zero:
this is the free response.

In [ ]:
msd = MassSpringDamper()
traj = msd.compute_trajectory(tf=10.0)
msd.plot_trajectory()

Every constant lives in `params`, so a new value needs no new class. More damping:

In [ ]:
msd.params["c"] = 2.0
msd.compute_trajectory(tf=10.0)
msd.plot_trajectory()

## 2. A catalog plant works the same way

minilink ships a catalog of ready-made models: `Pendulum`, `CartPole`, `DoubleIntegrator`,
`Drone2D`, manipulators, cars and more. They are the same kind of object as the class
above, so the same three lines simulate them. Each one also knows how to draw itself, so
`animate` plays the motion.

In [ ]:
from minilink import Pendulum

pendulum = Pendulum()
pendulum.params

In [ ]:
pendulum.x0 = np.array([2.0, 0.0])  # start at 2 rad, at rest
pendulum.params["d"] = 0.5  # add joint damping
pendulum.compute_trajectory(tf=10.0)
pendulum.plot_trajectory()

In [ ]:
pendulum.animate()

## 3. Connect a controller with `@`

`controller @ plant` closes the feedback loop: the plant's output `y` goes back to the
controller, and the controller's command `u` drives the plant. `Impedance` is a virtual
spring-damper on the joint, $u = K_p (r - q) - K_d \dot q$. The loop is a new system,
with the reference `r` as its input and the plant's `y` as its output.

In [ ]:
from minilink import Impedance

controller = Impedance(Kp=50.0, Kd=10.0)
loop = controller @ pendulum
loop

`closed_loop` builds the same loop and also drives its inputs. Here a `Step` source moves
the reference from 0 to 1 rad at $t = 2$ s. `plot_diagram` draws the blocks and their
wires.

In [ ]:
from minilink import Step, closed_loop

reference = Step(final_value=[1.0], step_time=2.0)
loop = closed_loop(controller, pendulum, reference=reference)
loop.plot_diagram()

## 4. Simulate, plot and animate the loop

A loop is a system like any other: the same calls simulate, plot and animate it. The
pendulum starts at 2 rad, is pulled back to 0, then follows the step to 1 rad.

In [ ]:
loop.compute_trajectory(tf=10.0)
loop.plot_trajectory()

In [ ]:
loop.animate()

Your own model connects in exactly the same way. A `PID` controller on the
mass-spring-damper of section 1, asked to hold the mass at $q = 0.5$:

In [ ]:
from minilink import PID

pid = PID(Kp=20.0, Ki=10.0, Kd=4.0)
msd_loop = closed_loop(pid, msd, reference=Step(final_value=[0.5], step_time=1.0))
msd_loop.compute_trajectory(tf=10.0)
msd_loop.plot_trajectory()

## 5. Animation options

`animate` takes a `renderer`: `"matplotlib"` (the default), `"plotly"` for an
interactive figure in the browser, `"meshcat"` for 3-D scenes and `"pygame"` for a
window on your own machine. The camera is set on the system before the call:
`camera_scale` is the half-width of the view, `camera_target` the point it looks at.

In [ ]:
pendulum.camera_scale = 3.0
pendulum.animate(renderer="plotly")

## Where to go next

You can now model, connect and simulate any system: write `f` and `h`, or take a plant
from the catalog, close a loop with `@`, and call `compute_trajectory`, `plot_trajectory`
and `animate`.

- [02_analyze_and_control](02_analyze_and_control.ipynb): linearize, read the poles,
  Bode and root locus, and design PID, LQR and LQG controllers.
- [03_plan_and_sample](03_plan_and_sample.ipynb): pose a planning problem, solve it, and
  run a digital controller on a clock.
- [core/](../core/01_systems.ipynb): the objects underneath, for the full picture.